In [170]:
import random
import time
from IPython.display import clear_output

In [72]:
class Cell:
    def __init__(self, x, y):
        self.x = x
        self.y = y
        self.walls = [True, True, True, True]  # top, right, bottom, left
        self.visited = False
        self.is_start = False
        self.is_end = False

class Maze:
    def __init__(self, width, height, is_image=False):
        self.width = width
        self.height = height
        self.start = None
        self.end = None
        self.grid = [[Cell(x, y) for y in range(height)] for x in range(width)]
        if not is_image:
            self.generate()

    def generate(self):
        stack = []
        current_cell = random.choice(random.choice(self.grid))
        current_cell.visited = True

        while True:
            neighbors = self.get_unvisited_neighbors(current_cell)
            if neighbors:
                next_cell = random.choice(neighbors)
                self.remove_wall(current_cell, next_cell)
                stack.append(current_cell)
                current_cell = next_cell
                current_cell.visited = True
            elif stack:
                current_cell = stack.pop()
            else:
                break

        for x in range(self.width):
            for y in range(self.height):
                if random.random() < 0.1:
                    cell = self.grid[x][y]
                    directions = [(0, -1), (1, 0), (0, 1), (-1, 0)]
                    valid = [(dx, dy, i) for i, (dx, dy) in enumerate(directions)
                             if 0 <= x + dx < self.width and 0 <= y + dy < self.height]
                    if valid:
                        dx, dy, wall_idx = random.choice(valid)
                        neighbor = self.grid[x + dx][y + dy]
                        self.remove_wall(cell, neighbor)

        min_dist = (self.width + self.height) / 2
        all_cells = [self.grid[x][y] for x in range(self.width) for y in range(self.height)]
        while True:
            self.start, self.end = random.sample(all_cells, 2)
            if abs(self.start.x - self.end.x) + abs(self.start.y - self.end.y) >= min_dist:
                break
        self.start.is_start = True
        self.end.is_end = True

    def get_unvisited_neighbors(self, cell):
        neighbors = []
        directions = [(0, -1), (1, 0), (0, 1), (-1, 0)]  # top, right, bottom, left
        for i, (dx, dy) in enumerate(directions):
            nx, ny = cell.x + dx, cell.y + dy
            if 0 <= nx < self.width and 0 <= ny < self.height:
                neighbor = self.grid[nx][ny]
                if not neighbor.visited:
                    neighbors.append(neighbor)
        return neighbors

    def remove_wall(self, current, next):
        dx = next.x - current.x
        dy = next.y - current.y
        if dx == 1:  # next is to the right
            current.walls[1] = False
            next.walls[3] = False
        elif dx == -1:  # next is to the left
            current.walls[3] = False
            next.walls[1] = False
        elif dy == 1:  # next is below
            current.walls[2] = False
            next.walls[0] = False
        elif dy == -1:  # next is above
            current.walls[0] = False
            next.walls[2] = False

    def display(self):
        for y in range(self.height):
            # Print the top walls
            for x in range(self.width):
                if self.grid[x][y].walls[0]:
                    print("+---", end="")
                else:
                    print("+   ", end="")
            print("+")
            # Print the left walls and cell contents
            for x in range(self.width):
                wall = "|" if self.grid[x][y].walls[3] else " "
                cell = self.grid[x][y]
                if cell.is_start:
                    print(f"{wall} O ", end="")
                elif cell is self.end:
                    print(f"{wall} X ", end="")
                else:
                    print(f"{wall}   ", end="")
            print("|")
        # Print the bottom walls of the last row
        for x in range(self.width):
            if self.grid[x][self.height - 1].walls[2]:
                print("+---", end="")
            else:
                print("+   ", end="")
        print("+")

In [185]:
class Agent:
    def __init__(self, maze):
        self.maze = maze
        self.position = maze.start
        self.img_x = 0
        self.img_y = 0
        self.maze_img = Maze(1, 1, is_image=True)
        self.maze_img.grid[0][0].walls = list(self.maze.start.walls)
        self.maze_img.grid[0][0].is_start = True
        self.maze_img.grid[0][0].visited = True

    def random_explore(self, steps=25):
        for step in range(steps):
            valid_directions = [i for i, w in enumerate(self.position.walls) if not w]
            next_direction = random.choice(valid_directions)
            dx, dy = [(0, -1), (1, 0), (0, 1), (-1, 0)][next_direction]

            # Expand image if agent is at the edge in that direction
            if next_direction == 0 and self.img_y == 0:
                self.add_row_or_col_to_img(0)
                self.reveal_shared_walls(0)
                self.img_y += 1
            elif next_direction == 1 and self.img_x == self.maze_img.width - 1:
                self.add_row_or_col_to_img(1)
                self.reveal_shared_walls(1)
            elif next_direction == 2 and self.img_y == self.maze_img.height - 1:
                self.add_row_or_col_to_img(2)
                self.reveal_shared_walls(2)
            elif next_direction == 3 and self.img_x == 0:
                self.add_row_or_col_to_img(3)
                self.reveal_shared_walls(3)
                self.img_x += 1

            # Move in actual maze and image
            self.position = self.maze.grid[self.position.x + dx][self.position.y + dy]
            self.img_x += dx
            self.img_y += dy

            # Reveal the new cell in the image
            img_cell = self.maze_img.grid[self.img_x][self.img_y]
            if not img_cell.visited:
                img_cell.walls = list(self.position.walls)
                img_cell.visited = True
                img_cell.is_start = self.position.is_start
                img_cell.is_end = self.position.is_end
                # Propagate open passages to already-existing neighbors in the image
                for di, (ndx, ndy) in enumerate([(0, -1), (1, 0), (0, 1), (-1, 0)]):
                    nx, ny = self.img_x + ndx, self.img_y + ndy
                    if 0 <= nx < self.maze_img.width and 0 <= ny < self.maze_img.height:
                        if not img_cell.walls[di]:
                            opposite = [2, 3, 0, 1][di]
                            self.maze_img.grid[nx][ny].walls[opposite] = False

            clear_output(wait=True)
            print(f"Step {step + 1}/{steps}")
            self.display_maze_img()
            time.sleep(0.15)

    def reveal_shared_walls(self, direction):
        if direction == 0:  # New row at top (y=0); existing cells now at y=1+
            for x in range(self.maze_img.width):
                neighbor = self.maze_img.grid[x][1]
                if neighbor.visited and not neighbor.walls[0]:
                    self.maze_img.grid[x][0].walls[2] = False
        elif direction == 1:  # New column at right (x=width-1)
            for y in range(self.maze_img.height):
                neighbor = self.maze_img.grid[self.maze_img.width - 2][y]
                if neighbor.visited and not neighbor.walls[1]:
                    self.maze_img.grid[self.maze_img.width - 1][y].walls[3] = False
        elif direction == 2:  # New row at bottom (y=height-1)
            for x in range(self.maze_img.width):
                neighbor = self.maze_img.grid[x][self.maze_img.height - 2]
                if neighbor.visited and not neighbor.walls[2]:
                    self.maze_img.grid[x][self.maze_img.height - 1].walls[0] = False
        elif direction == 3:  # New column at left (x=0); existing cells now at x=1+
            for y in range(self.maze_img.height):
                neighbor = self.maze_img.grid[1][y]
                if neighbor.visited and not neighbor.walls[3]:
                    self.maze_img.grid[0][y].walls[1] = False

    def add_row_or_col_to_img(self, direction):
        if direction == 0:  # Add row at top
            for col in self.maze_img.grid:
                col.insert(0, Cell(0, 0))
            self.maze_img.height += 1
        elif direction == 1:  # Add column at right
            self.maze_img.grid.append([Cell(0, 0) for _ in range(self.maze_img.height)])
            self.maze_img.width += 1
        elif direction == 2:  # Add row at bottom
            for col in self.maze_img.grid:
                col.append(Cell(0, 0))
            self.maze_img.height += 1
        elif direction == 3:  # Add column at left
            self.maze_img.grid.insert(0, [Cell(0, 0) for _ in range(self.maze_img.height)])
            self.maze_img.width += 1

        for x in range(self.maze_img.width):
            for y in range(self.maze_img.height):
                self.maze_img.grid[x][y].x = x
                self.maze_img.grid[x][y].y = y

    def display_maze_img(self):
        for y in range(self.maze_img.height):
            for x in range(self.maze_img.width):
                if self.maze_img.grid[x][y].walls[0]:
                    print("+---", end="")
                else:
                    print("+   ", end="")
            print("+")
            for x in range(self.maze_img.width):
                wall = "|" if self.maze_img.grid[x][y].walls[3] else " "
                cell = self.maze_img.grid[x][y]
                if x == self.img_x and y == self.img_y:
                    print(f"{wall} A ", end="")
                elif cell.is_start:
                    print(f"{wall} O ", end="")
                elif cell.is_end:
                    if not cell.visited:
                        print(f"{wall} ? ", end="")
                    else:
                        print(f"{wall} X ", end="")
                elif not cell.visited:
                    print(f"{wall} ? ", end="")
                else:
                    print(f"{wall}   ", end="")
            right_wall = "|" if self.maze_img.grid[self.maze_img.width - 1][y].walls[1] else " "
            print(right_wall)
        for x in range(self.maze_img.width):
            if self.maze_img.grid[x][self.maze_img.height - 1].walls[2]:
                print("+---", end="")
            else:
                print("+   ", end="")
        print("+")


In [192]:
maze = Maze(8, 8)
maze.display()

+---+---+---+---+---+---+---+---+
|   |               |       |   |
+   +   +   +---+   +   +   +   +
|       |       |   |   |   |   |
+---+---+---+   +   +   +   +   +
|                 O |       |   |
+   +---+---+---+   +   +   +   +
|       |   |           |       |
+---+   +   +   +---+---+---+   +
|       |       |   |   |   |   |
+   +---+   +---+   +   +   +   +
|   |               |   |       |
+   +---+---+---+   +   +---+   +
|       |           |       |   |
+   +   +---+---+---+---+   +   +
| X |                       |   |
+---+---+---+---+---+---+---+---+


In [193]:
agent = Agent(maze)
agent.display_maze_img()

+   +
  A |
+   +


In [195]:
agent.random_explore(steps=300)

Step 147/300
+---+---+---+---+---+---+---+---+
| ? | ? | ? | ?     |       |   |
+---+---+---+---+   +   +   +   +
| ? | ? | ? | ? |   |   |   |   |
+---+---+---+   +   +   +   +   +
|                 O |       |   |
+   +---+---+---+   +   +   +   +
|       |   |           |       |
+---+   +   +   +---+---+---+   +
|       |       | ? | A |   |   |
+   +---+   +---+---+   +   +   +
|   |         ? | ? |   |       |
+   +---+---+---+---+   +---+   +
|       | ? | ? | ? |       |   |
+   +   +---+---+---+---+   +   +
| X |                       |   |
+---+---+---+---+---+---+---+---+


KeyboardInterrupt: 